# Run the ML batch jobs

Runs `anomaly_detection.py` (Isolation Forest), `usage_profiling.py`
(KMeans), and `forecasting.py` (Prophet) against whatever data currently
exists — mirrors the old standalone-script pattern, except now each model
lives in its own file so it can be scheduled independently in production
(e.g. anomaly detection every 4 hours via cron/systemd timer, usage
profiling every 3 months, forecasting monthly). This notebook is only for
manual/ad-hoc "run everything now" use during development.

BackendDotNet must already exist and have its base data loaded first:

```
cd BackendDotNet && dotnet ef database update && dotnet run -- --seed
```

If its database isn't reachable yet, the run is skipped with a warning
rather than failing.

In [ ]:
# import the anomaly-detection module (Isolation Forest job)
import anomaly_detection
# import the usage-profiling module (KMeans job)
import usage_profiling
# import the forecasting module (Prophet job)
import forecasting
# import the billing module (call-cost computation job)
import billing
# import the generator that yields a live connection to every reachable backend
from common import open_backend_connections


## Run

In [ ]:
# dict that will collect each backend's result summary, keyed by backend name
summary = {}
# loop over every currently-reachable backend (today, just BackendDotNet)
for backend in open_backend_connections():
    # run each of the four batch jobs in turn against this backend, and
    # collect their return values (rows written/flags created/etc) into one dict
    result = {
        # recompute call costs from the CDR dataset -> usage_summaries
        "usage_summaries_written": billing.run(backend),
        # fit Isolation Forest and flag new anomalous usage_events
        "anomalies_flagged": anomaly_detection.run(backend),
        # fit KMeans and update each employee's usage cluster label
        "usage_profiles_updated": usage_profiling.run(backend),
        # fit Prophet (or the naive fallback) and write next-month forecasts
        "forecasts_written": forecasting.run(backend),
    }
    # store this backend's results under its name in the summary dict
    summary[backend.name] = result

    # print a short human-readable report for this backend
    print(f"[{backend.name}]")
    print(f"  Billing:          {result['usage_summaries_written']} usage_summary row(s) written.")
    print(f"  Isolation Forest: {result['anomalies_flagged']} new anomaly flag(s) created.")
    print(f"  KMeans:           {result['usage_profiles_updated']} employee usage profile(s) updated.")
    print(f"  Forecasting:      {result['forecasts_written']} department forecast(s) written.")

# display the full summary dict as this cell's output too
summary